# Sparkle V2 — Fase 3.8: métricas derivadas y reglas

**Protocolo pre-registrado:** `Protocolo_Fase38_Metricas_Derivadas.md` (2026-08-07).

El modelo actual consume media y desvío temporal de 22 características. Ese resumen descarta
la estructura temporal *dentro* de la ventana: no distingue un cierre ocular sostenido de
parpadeos frecuentes, ni una desviación breve de la cabeza de una prolongada.

Este notebook calcula las métricas derivadas que la literatura de fatiga y desatención emplea
—PERCLOS, tasa de parpadeo, duración de bostezo, persistencia cefálica— **sobre los clips ya
extraídos**, sin descargar datos nuevos ni reprocesar video, y mide si aportan.

**Umbrales congelados en §3 del protocolo.** No se ajustan tras ver resultados.

**`roll` no está disponible**: la extracción guarda solo `yaw` y `pitch`. Incorporarlo exige
re-extraer (~1 h 45) y queda fuera de esta fase.

**El Test no se toca.**

In [ ]:
import sys, os, json, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

CACHE_32  = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
CACHE_OLD = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_5_raw1280.npz'
PROJ_ORIG = f'{PROJECT_DIR}/checkpoints_fase3_2/proyeccion_aleatoria_original.npz'
OUT       = f'{PROJECT_DIR}/fase3_8_metricas'
os.makedirs(OUT, exist_ok=True)

import numpy as np, pandas as pd
from sklearn.metrics import balanced_accuracy_score, accuracy_score, roc_auc_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# ------- CONGELADOS por el protocolo §3 -------
SEED, UMBRAL, N_FOLDS, N_BOOT = 42, 0.50, 5, 5000
D_PROJ, N_PROY, DIM_RAW = 16, 8, 1280
N_FRAMES, FPS = 60, 6.0
PERCLOS_BASE_PCT, PERCLOS_FRAC = 75, 0.80   # base = p75 del sujeto; umbral = 80 % de esa base
BLINK_TH   = 0.5
JAW_TH     = 0.5
YAW_TH     = 15.0     # grados
PITCH_TH   = -10.0    # grados (mirar hacia abajo)
REGLA_TS   = [3, 5, 8]   # segundos, para el análisis 4
MARGEN_MIN = 0.02

# Orden de las 22 características (fijo desde Fase 2)
FEATS = ['ear_avg', 'ear_left', 'ear_right', 'yaw', 'pitch', 'mar',
         'eyeBlinkLeft', 'eyeBlinkRight', 'eyeSquintLeft', 'eyeSquintRight',
         'eyeLookInLeft', 'eyeLookOutLeft', 'eyeLookUpLeft', 'eyeLookDownLeft',
         'eyeLookInRight', 'eyeLookOutRight', 'eyeLookUpRight', 'eyeLookDownRight',
         'browDownLeft', 'browDownRight', 'browInnerUp', 'jawOpen']
IX = {n: i for i, n in enumerate(FEATS)}
BLENDSHAPES = list(range(6, 22))   # los 16 blendshapes

d32 = np.load(CACHE_32, allow_pickle=True)
assert [str(f) for f in d32['feature_names']] == FEATS, 'El orden de features no coincide'
Xg_pool = np.concatenate([d32['Xg_tr'], d32['Xg_va']])          # (N, 60, 22) CRUDAS, con NaN
y_pool  = (np.concatenate([d32['y3_tr'], d32['y3_va']]) > 0).astype(np.int64)
id_pool = np.array([str(c) for c in np.concatenate([d32['id_tr'], d32['id_va']])])
part_pool = np.array([c[:6] for c in id_pool])
print(f'Pool: {Xg_pool.shape[0]} clips · {len(set(part_pool))} participantes · '
      f'{Xg_pool.shape[1]} cuadros × {Xg_pool.shape[2]} features')

## 1. Cálculo de las métricas derivadas

Todas operan sobre la secuencia CRUDA de 60×22, tratando los NaN (cuadros sin rostro) como
ausencia de evento en lugar de propagarlos.

In [ ]:
def rachas_max(mask):
    """Longitud de la racha de `True` más larga, por fila. mask: (N, T) booleano."""
    N, T = mask.shape
    mejor = np.zeros(N, dtype=np.int32); actual = np.zeros(N, dtype=np.int32)
    for t in range(T):
        actual = np.where(mask[:, t], actual + 1, 0)
        mejor = np.maximum(mejor, actual)
    return mejor

def cuenta_eventos(mask):
    """Cantidad de rachas maximales de `True` por fila (transiciones False→True)."""
    prev = np.concatenate([np.zeros((mask.shape[0], 1), bool), mask[:, :-1]], axis=1)
    return (mask & ~prev).sum(axis=1)

def calcular_metricas(X, participantes):
    """X: (N,60,22) crudas. Devuelve DataFrame con las 10 métricas del protocolo §3."""
    N, T, _ = X.shape
    val = ~np.isnan(X[:, :, 0])                      # cuadros con rostro

    # --- 1. PERCLOS con línea base POR SUJETO ---
    ear = X[:, :, IX['ear_avg']]
    base = np.full(N, np.nan)
    for p in np.unique(participantes):
        m = participantes == p
        vals = ear[m][~np.isnan(ear[m])]
        base[m] = np.percentile(vals, PERCLOS_BASE_PCT) if vals.size else np.nan
    umbral_ear = base[:, None] * PERCLOS_FRAC
    cerrado = (ear < umbral_ear) & val
    perclos = cerrado.sum(axis=1) / np.maximum(val.sum(axis=1), 1)

    # --- 2. Tasa de parpadeo (eventos por minuto) ---
    blink = np.nanmean(X[:, :, [IX['eyeBlinkLeft'], IX['eyeBlinkRight']]], axis=2)
    mblink = np.nan_to_num(blink) > BLINK_TH
    tasa_parpadeo = cuenta_eventos(mblink) * (60.0 / (T / FPS))

    # --- 3. Boca abierta: racha más larga en segundos ---
    mjaw = np.nan_to_num(X[:, :, IX['jawOpen']]) > JAW_TH
    boca_abierta_max = rachas_max(mjaw) / FPS

    # --- 4/5. Desviación de yaw ---
    myaw = (np.abs(np.nan_to_num(X[:, :, IX['yaw']])) > YAW_TH) & val
    yaw_desviado = myaw.sum(axis=1) / np.maximum(val.sum(axis=1), 1)
    yaw_racha_max = rachas_max(myaw) / FPS

    # --- 6/7. Mirar hacia abajo ---
    mp = (np.nan_to_num(X[:, :, IX['pitch']]) < PITCH_TH) & val
    pitch_abajo = mp.sum(axis=1) / np.maximum(val.sum(axis=1), 1)
    pitch_racha_max = rachas_max(mp) / FPS

    # --- 8. Reactividad facial: cambio absoluto medio cuadro a cuadro ---
    bs = X[:, :, BLENDSHAPES]
    reactividad = np.nanmean(np.abs(np.diff(bs, axis=1)), axis=(1, 2))

    # --- 9. Mirada hacia afuera ---
    fuera = np.nanmax(X[:, :, [IX['eyeLookOutLeft'], IX['eyeLookOutRight']]], axis=2)
    mirada_fuera = np.nanmean(fuera, axis=1)

    # --- 10. Ojos entrecerrados ---
    ojos_entrecerrados = np.nanmean(
        X[:, :, [IX['eyeSquintLeft'], IX['eyeSquintRight']]], axis=(1, 2))

    df = pd.DataFrame({
        'perclos': perclos, 'tasa_parpadeo': tasa_parpadeo,
        'boca_abierta_max': boca_abierta_max,
        'yaw_desviado': yaw_desviado, 'yaw_racha_max': yaw_racha_max,
        'pitch_abajo': pitch_abajo, 'pitch_racha_max': pitch_racha_max,
        'reactividad': reactividad, 'mirada_fuera': mirada_fuera,
        'ojos_entrecerrados': ojos_entrecerrados})
    return df.fillna(df.median())

t0 = time.time()
M = calcular_metricas(Xg_pool, part_pool)
print(f'{M.shape[1]} métricas sobre {M.shape[0]} clips en {time.time()-t0:.1f} s\n')
print(M.describe().T[['mean', 'std', 'min', '50%', 'max']].round(4).to_string())

## 2. Análisis 1 — Inventario univariado

AUC de cada métrica como predictor individual. **Descriptivo: no habilita incorporar nada**
(§5 del protocolo). Con 10 comparaciones, la probabilidad de al menos un falso positivo al
5 % ronda el 40 %.

In [ ]:
def boot_auc(v, y, part, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}
    out = []
    for _ in range(n_boot):
        j = np.concatenate([idx[q] for q in rng.choice(parts, size=len(parts), replace=True)])
        if len(np.unique(y[j])) < 2:
            continue
        out.append(roc_auc_score(y[j], v[j]))
    return [round(float(x), 4) for x in np.percentile(out, [2.5, 97.5])]

filas = []
for col in M.columns:
    v = M[col].values
    auc = roc_auc_score(y_pool, v)
    # AUC < 0,5 significa que predice en sentido inverso; se reporta la magnitud del efecto
    filas.append({'metrica': col, 'AUC': round(auc, 4),
                  'AUC_IC95': boot_auc(v, y_pool, part_pool),
                  'separacion': round(abs(auc - 0.5), 4)})
inv = pd.DataFrame(filas).sort_values('separacion', ascending=False)
print(inv.to_string(index=False))
print('\nAUC = 0,5 significa ausencia de señal. Valores por debajo indican relación inversa.')
inv.to_csv(f'{OUT}/inventario_univariado.csv', index=False)

## 3. Análisis 3 — Redundancia con el resumen actual

El modelo ya recibe media y desvío de las 22 características. Una métrica muy correlacionada
con el desvío de su característica de origen es redundante por construcción.

In [ ]:
def resumir(X):
    X = np.nan_to_num(X); return np.hstack([X.mean(axis=1), X.std(axis=1)])

R22 = resumir(Xg_pool)     # 44 columnas: 22 medias + 22 desvíos
nombres_R = [f'media_{n}' for n in FEATS] + [f'desvio_{n}' for n in FEATS]
filas = []
for col in M.columns:
    c = np.array([abs(np.corrcoef(M[col].values, R22[:, k])[0, 1]) for k in range(R22.shape[1])])
    c = np.nan_to_num(c)
    k = int(np.argmax(c))
    filas.append({'metrica': col, 'corr_max': round(float(c[k]), 3), 'con': nombres_R[k]})
red = pd.DataFrame(filas).sort_values('corr_max', ascending=False)
print(red.to_string(index=False))
print('\nCorrelación por encima de 0,9 indica redundancia casi total con lo que el modelo ya ve.')
red.to_csv(f'{OUT}/redundancia.csv', index=False)

## 4. Análisis 2 — Aporte incremental (el que decide)

Brazo base: la configuración validada. Brazo ampliado: lo mismo más las 10 métricas.
Es **una sola comparación**, no diez, de modo que no requiere corrección por multiplicidad.

In [ ]:
dold = np.load(CACHE_OLD, allow_pickle=True)
Wo = np.load(PROJ_ORIG); W_ORIG, B_ORIG = Wo['W'], Wo['b']
Xa_pool = np.concatenate([dold['Xr_tr'], dold['Xr_va']])
DELTA = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen', 'eyeBlinkLeft', 'eyeBlinkRight',
         'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']
delta_idx = [IX[f] for f in DELTA]

def interp(seq):
    df = pd.DataFrame(seq).interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean(X, medians=None, thresh=0.30):
    keep, out = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > thresh:
            continue
        s = interp(X[i])
        if np.isnan(s).any() and medians is not None:
            ix = np.where(np.isnan(s)); s[ix] = np.take(medians, ix[1])
        out.append(s); keep.append(i)
    return np.stack(out), np.array(keep)

def add_deltas(X):
    d = np.zeros_like(X[:, :, delta_idx]); d[:, 1:, :] = np.diff(X[:, :, delta_idx], axis=1)
    return np.concatenate([X, d], axis=-1)

def _ba(y, ok):
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    return np.nan if (n0 == 0 or n1 == 0) else 0.5*(ok[y==0].sum()/n0 + ok[y==1].sum()/n1)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(sgkf.split(Xg_pool, y_pool, groups=part_pool))
Mv = M.values

def correr_oof(con_metricas):
    oof = np.full(len(y_pool), np.nan); mask = np.zeros(len(y_pool), bool)
    for fold, (tr, va) in enumerate(folds):
        assert not (set(part_pool[tr]) & set(part_pool[va])), f'LEAKAGE fold {fold}'
        g_med = np.nanmedian(Xg_pool[tr].reshape(-1, 22), axis=0)
        Xa_, kga = clean(Xg_pool[tr], g_med); Xb_, kgb = clean(Xg_pool[va], g_med)
        Ea, kea = clean(Xa_pool[tr].astype(np.float32), None, thresh=0.5)
        Eb, keb = clean(Xa_pool[va].astype(np.float32), None, thresh=0.5)
        ka = np.intersect1d(kga, kea); kb = np.intersect1d(kgb, keb)
        ig = {k: i for i, k in enumerate(kga)}; ie = {k: i for i, k in enumerate(kea)}
        jg = {k: i for i, k in enumerate(kgb)}; je = {k: i for i, k in enumerate(keb)}
        gsc = StandardScaler().fit(add_deltas(Xa_).reshape(-1, 32))
        sg = lambda X: gsc.transform(X.reshape(-1, 32)).reshape(len(X), N_FRAMES, 32)
        Rg_tr = resumir(sg(add_deltas(Xa_))[[ig[k] for k in ka]])
        Rg_ev = resumir(sg(add_deltas(Xb_))[[jg[k] for k in kb]])
        Ea_k = Ea[[ie[k] for k in ka]]; Eb_k = Eb[[je[k] for k in kb]]
        y_tr = y_pool[tr][ka]; idx = va[kb]; mask[idx] = True
        Mtr = Mv[tr][ka]; Mev = Mv[va][kb]
        ps = []
        for j in range(N_PROY):
            cols = slice(j*D_PROJ, (j+1)*D_PROJ)
            Rt = np.hstack([Rg_tr, resumir(Ea_k @ W_ORIG[:, cols] + B_ORIG[cols])])
            Re = np.hstack([Rg_ev, resumir(Eb_k @ W_ORIG[:, cols] + B_ORIG[cols])])
            if con_metricas:
                Rt = np.hstack([Rt, Mtr]); Re = np.hstack([Re, Mev])
            sc = StandardScaler().fit(Rt)
            lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
            lr.fit(sc.transform(Rt), y_tr)
            ps.append(lr.predict_proba(sc.transform(Re))[:, 1])
        oof[idx] = np.mean(ps, axis=0)
    return oof, mask

t0 = time.time()
p_base, mask = correr_oof(False)
p_ext, mask2 = correr_oof(True)
assert (mask == mask2).all()
y_c = y_pool[mask]; part_c = part_pool[mask]
p_base = p_base[mask]; p_ext = p_ext[mask]
print(f'OOF listo en {(time.time()-t0)/60:.1f} min · {mask.sum()} clips\n')

def met(y, p):
    pred = (p > UMBRAL).astype(int)
    return {'accuracy': round(accuracy_score(y, pred), 4),
            'balanced_acc': round(balanced_accuracy_score(y, pred), 4),
            'f1_macro': round(f1_score(y, pred, average='macro'), 4),
            'auc': round(roc_auc_score(y, p), 4)}

print('BASE     :', met(y_c, p_base))
print('AMPLIADO :', met(y_c, p_ext))

def boot_pareado(pa, pb, y=None, part=None, n_boot=N_BOOT, seed=SEED):
    y = y_c if y is None else y; part = part_c if part is None else part
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}
    oka = (pa > UMBRAL).astype(int) == y; okb = (pb > UMBRAL).astype(int) == y
    d = np.empty(n_boot)
    for i in range(n_boot):
        j = np.concatenate([idx[q] for q in rng.choice(parts, size=len(parts), replace=True)])
        d[i] = _ba(y[j], okb[j]) - _ba(y[j], oka[j])
    d = d[~np.isnan(d)]; lo, hi = np.percentile(d, [2.5, 97.5])
    return {'delta_balacc': round(float(d.mean()), 4),
            'IC95': [round(float(lo), 4), round(float(hi), 4)], 'cruza_cero': bool(lo <= 0 <= hi)}

r_inc = boot_pareado(p_base, p_ext)
print('\nAmpliado − base:', r_inc)

## 5. Análisis 4 — Reglas de persistencia como anulación

«Si la racha supera *T* segundos, forzar desatención», con *T* fijado a priori.
Se aplica sobre la decisión del brazo base.

In [ ]:
idx_ok = np.where(mask)[0]
filas = []
for nombre in ['yaw_racha_max', 'pitch_racha_max']:
    v = Mv[idx_ok, list(M.columns).index(nombre)]
    for T in REGLA_TS:
        p_regla = np.where(v > T, 1.0, p_base)      # forzar desatención
        r = boot_pareado(p_base, p_regla)
        filas.append({'regla': f'{nombre} > {T}s', 'clips_afectados': int((v > T).sum()),
                      'pct_afectados': round(float((v > T).mean()), 4),
                      **met(y_c, p_regla), **r})
reglas = pd.DataFrame(filas)
print(reglas[['regla', 'pct_afectados', 'balanced_acc', 'delta_balacc', 'IC95',
              'cruza_cero']].to_string(index=False))
reglas.to_csv(f'{OUT}/reglas.csv', index=False)

## 6. Veredicto contra el criterio pre-registrado

In [ ]:
print('=' * 68)
print(f'BASE     — balanced accuracy {met(y_c, p_base)["balanced_acc"]:.4f}')
print(f'AMPLIADO — balanced accuracy {met(y_c, p_ext)["balanced_acc"]:.4f}')
print(f'Diferencia: {r_inc["delta_balacc"]:+.4f}  IC95 {r_inc["IC95"]}')
print('=' * 68)
if r_inc['cruza_cero'] or r_inc['delta_balacc'] < MARGEN_MIN:
    print('❌ Las métricas derivadas NO aportan de forma demostrable.')
    print('   El criterio exige IC que no cruce cero Y mejora ≥ +2,0 pts.')
    print('   Conclusión pre-registrada: el resumen por media y desvío ya captura la señal')
    print('   temporal disponible. Se documenta como resultado negativo (regla 10) y NO se')
    print('   modifica el modelo desplegado.')
else:
    print('✅ Las métricas derivadas aportan por encima del umbral de relevancia.')
    print('   Requiere: nuevo pre-registro, reentrenamiento del artefacto y una evaluación')
    print('   de cierre sobre Test con su propia enmienda.')
buenas = reglas[(~reglas.cruza_cero) & (reglas.delta_balacc >= MARGEN_MIN)]
print(f'\nReglas de anulación que cumplen el criterio: {len(buenas)} de {len(reglas)}')
if len(buenas):
    print(buenas[['regla', 'delta_balacc', 'IC95']].to_string(index=False))

json.dump({'inventario': inv.to_dict('records'), 'redundancia': red.to_dict('records'),
           'base': met(y_c, p_base), 'ampliado': met(y_c, p_ext), 'incremental': r_inc,
           'reglas': reglas.to_dict('records')},
          open(f'{OUT}/fase3_8.json', 'w'), indent=2, ensure_ascii=False, default=str)
print(f'\nGuardado en {OUT}/. Evaluaciones sobre Test: 0.')

## 7. Pendiente declarado: el ángulo `roll`

No se evalúa acá porque la extracción guarda solo `yaw` y `pitch`. Incorporarlo requiere
modificar la obtención de ángulos de Euler en **ambos** lenguajes y re-extraer el dataset.

En Python, `euler_from_transformation_matrix` pasaría a devolver también:

```python
roll = math.degrees(math.atan2(rot[1, 0], rot[0, 0]))
```

y en `features.js`, dentro de `eulerFromMatrix`, el equivalente con el mismo acceso
column-major ya verificado. La correspondencia entre ambas implementaciones debe comprobarse
con el mismo procedimiento que se aplicó a `yaw` y `pitch`.

Justificación para atacarlo solo si esta fase resulta positiva: `roll` es el eje asociado a
apoyar la cabeza sobre la mano, gesto que la literatura vincula con fatiga. Si ninguna de las
diez métricas de persistencia y oculometría aporta, es poco probable que un undécimo ángulo
lo haga, y el costo es de ~1 h 45 de re-extracción más la revalidación completa.